# INFO8665 Local Orchestrator Notebook

This notebook orchestrates the **local-first** workflow for the SeeMyCash currency-recognition pipeline:

1. Download Roboflow Universe datasets (optional; requires `ROBOFLOW_API_KEY`).
2. Merge/normalize banknotes + coins into canonical detector datasets.
3. Build a negatives pool (confusers + backgrounds).
4. Build bill cutouts (split-preserving ImageFolder).
5. Build coin crop domains (synthetic vs real).
6. Camera-realism augmentation for coin domains.
7. Lightweight EDA summary.
8. Train detector (YOLO26), coin classifier, bill classifier.
9. Evaluate classifiers + export artifacts (ONNX / INT8 / TorchScript).
10. Train & evaluate spoof-guard classifier.
11. Prepare screen-guard dataset & submit SageMaker training job.
12. Download champion models from SageMaker.
13. Run Docker Compose inference API + React Native web UI.

All outputs are written under `data/` and `outputs/` (both are git-ignored).
Trained models live in `training/models/`.

## 0) Setup

This notebook expects to be run from the `INFO8665/` repo root.

### Local Python
```bash
pip install -r requirements.txt
```

### Docker (data pipeline image)
```bash
docker build -t info8665-local .
```

### Docker Compose (inference API + tests)
```bash
docker compose up api          # start inference API on :8080
docker compose --profile test up  # run E2E test suite against the API
```

### Environment variables

| Variable | Required | Purpose |
|----------|----------|---------|
| `ROBOFLOW_API_KEY` | Optional | Roboflow Universe dataset download |
| `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` | Optional | SageMaker training + model download |
| `ENABLE_AWS_MODEL_SYNC` | Optional | Set `1` to auto-sync champion models from S3 on API startup |

The pipeline still works without the download step if you already have `data/interim/...` populated.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from subprocess import run

REPO_ROOT = Path.cwd()
print("cwd=", REPO_ROOT)


def sh(cmd: list[str]) -> None:
    print("\n$", " ".join(cmd))
    run(cmd, check=False)


def has_env(name: str) -> bool:
    return bool(os.environ.get(name, "").strip())


print("has_ROBOFLOW_API_KEY=", has_env("ROBOFLOW_API_KEY"))

## 1) Download from Roboflow Universe (optional)

Downloads to `data/interim/<category>/...`.

Skip this section if you already have datasets under `data/interim/`.

In [ ]:
# Requires ROBOFLOW_API_KEY; this will exit if not set.
# Comment out if you want to skip download.
sh([
    sys.executable,
    "scripts/01_download_from_universe.py",
    "--links",
    "configs/universe_links.yaml",
    "--out",
    "data/interim",
    "--format",
    "yolov8",
])

## 2) Merge + normalize datasets

### 2a) Banknotes (for bill cutouts + bill classifier)

- Output: `data/processed/banknotes_merged/`
- Config: `configs/label_map.yaml`

### 2b) Money detector dataset (banknotes + coins)

If you want the detector to detect **banknotes and coins**, build:

- Output: `data/processed/money_merged/`
- Config: `configs/label_map_money.yaml` (adds `COIN`)

In [ ]:
# 2a) Banknotes merged
sh([
    sys.executable,
    "scripts/02_merge_and_normalize_banknotes.py",
    "--in-root",
    "data/interim/banknotes",
    "--label-map",
    "configs/label_map.yaml",
    "--out",
    "data/processed/banknotes_merged",
])

# 2b) Money merged (banknotes + coins) for a unified detector
sh([
    sys.executable,
    "scripts/02_merge_and_normalize_money.py",
    "--banknotes-root",
    "data/interim/banknotes",
    "--coins-root",
    "data/interim/coins",
    "--label-map",
    "configs/label_map_money.yaml",
    "--out",
    "data/processed/money_merged",
])

## 3) Negatives pool (confusers + backgrounds)

Build a deduplicated negative image pool:

- Output: `data/processed/negatives/images/`

In [ ]:
sh([
    sys.executable,
    "scripts/03_build_negatives_pool.py",
    "--confusers-root",
    "data/interim/confusers",
    "--backgrounds-root",
    "data/interim/backgrounds",
    "--out",
    "data/processed/negatives",
])

## 4) Build bill cutouts (split-preserving)

Creates RGBA PNG cutouts for bill classification:

- Output: `data/processed/bill_cutouts/{train,val,test}/<class>/*.png`

In [ ]:
sh([
    sys.executable,
    "scripts/04_make_bill_cutouts.py",
    "--dataset",
    "data/processed/banknotes_merged",
    "--out",
    "data/processed/bill_cutouts",
])

## 5) Build coin crop domains (synthetic vs real)

Creates ImageFolder domain splits for coin classifier domain adaptation:

- Output: `data/processed/coin_domains/{A_synth,B_real}/{train,val}/<class>/`

In [ ]:
sh([
    sys.executable,
    "scripts/20_build_coin_crops_domains.py",
    "--banknotes-root",
    "data/interim/banknotes",
    "--coins-root",
    "data/interim/coins",
    "--out",
    "data/processed/coin_domains",
])

## 6) Camera-realism augmentation for coin domains

Applies photometric and geometric augmentations that simulate real-world camera capture:

- Output: `data/processed/coin_domains_augmented/`

In [ ]:
sh([
    sys.executable,
    "scripts/21_camera_realism_augment.py",
    "--in-dir",
    "data/processed/coin_domains",
    "--out-dir",
    "data/processed/coin_domains_augmented",
])

## 7) Lightweight EDA summary

Writes a small JSON report with split counts + class balance.

In [ ]:
sh([
    sys.executable,
    "scripts/06_eda_banknotes_summary.py",
    "--dataset",
    "data/processed/banknotes_merged",
    "--out-json",
    "outputs/reports/banknotes_eda_summary.json",
])

## 8) Train detector (YOLO26)

Trains locally using Ultralytics. Outputs under `outputs/models/<run-name>/...`.

Prefers the unified `money_merged` dataset (banknotes + coins) if present; falls back to `banknotes_merged`.

In [ ]:
# Prefer the unified money_merged dataset if present; fall back to banknotes_merged.
money_yaml = Path("data/processed/money_merged/data.yaml")
banknotes_yaml = Path("data/processed/banknotes_merged/data.yaml")

train_yaml = str(money_yaml) if money_yaml.exists() else str(banknotes_yaml)
print("detector_data=", train_yaml)

sh([
    sys.executable,
    "scripts/10_train_detector.py",
    "--data",
    train_yaml,
    "--weights",
    "yolo26n.pt",
    "--imgsz",
    "640",
    "--epochs",
    "25",
    "--batch",
    "16",
    "--project",
    "outputs/models",
    "--run-name",
    "detector_local",
])

## 9) Export detector to ONNX

- Standard ONNX: does not require calibration
- INT8 ONNX: uses separate quantization script (step 10)

In [ ]:
# Export standard ONNX (no calibration)
# For INT8 calibrated ONNX, add: --int8 --data data/processed/banknotes_merged/data.yaml

trained_weights = Path("outputs/models/detector_local/weights/best.pt")
weights_for_export = str(trained_weights) if trained_weights.exists() else "yolo26n.pt"
print("weights_for_export=", weights_for_export)

sh([
    sys.executable,
    "scripts/11_export_detector_onnx.py",
    "--weights",
    weights_for_export,
    "--imgsz",
    "640",
    "--out",
    "outputs/models/detector_export.onnx",
])

## 10) Quantize detector to INT8 ONNX (optional)

Requires the calibration dataset to compute quantization parameters. Produces a smaller, faster ONNX model for CPU/edge inference.

In [ ]:
# Requires a calibration dataset (uses the same training data.yaml)
sh([
    sys.executable,
    "scripts/27_quantize_detector_onnx_int8.py",
    "--onnx-in",
    "outputs/models/detector_export.onnx",
    "--data",
    train_yaml,
    "--out",
    "outputs/models/detector_int8.onnx",
])

## 11) Train coin classifier (local)

Uses the coin crop domains from step 5 (or augmented domains from step 6).

- Output: `outputs/models/coin_classifier_local.pt`

In [ ]:
# Use augmented domains if available; fall back to raw domains
coin_train = "data/processed/coin_domains_augmented/A_synth"
coin_val = "data/processed/coin_domains/B_real"

if not Path(coin_train).exists():
    coin_train = "data/processed/coin_domains/A_synth"
    print("Using raw coin domains (no augmentation)")

sh([
    sys.executable,
    "scripts/22_train_coin_classifier.py",
    "--train-dir",
    coin_train,
    "--val-dir",
    coin_val,
    "--output",
    "outputs/models/coin_classifier_local.pt",
    "--epochs",
    "20",
    "--batch-size",
    "64",
    "--lr",
    "0.001",
    "--img-size",
    "256",
    "--backbone",
    "resnet18",
])

## 12) Train bill classifier (local)

Uses the split-preserving bill cutouts from step 4.

- Output: `outputs/models/bill_classifier_local.pt`

In [ ]:
sh([
    sys.executable,
    "scripts/24_train_bill_classifier.py",
    "--train-dir",
    "data/processed/bill_cutouts/train",
    "--val-dir",
    "data/processed/bill_cutouts/val",
    "--output",
    "outputs/models/bill_classifier_local.pt",
    "--epochs",
    "20",
    "--batch-size",
    "64",
    "--lr",
    "0.001",
    "--img-size",
    "256",
    "--backbone",
    "resnet18",
])

## 13) Evaluate classifiers

Run evaluation on the held-out test/val splits and generate metrics reports.

In [ ]:
# Evaluate coin classifier
sh([
    sys.executable,
    "scripts/23_eval_coin_classifier.py",
    "--checkpoint",
    "outputs/models/coin_classifier_local.pt",
    "--val-dir",
    coin_val,
])

# Evaluate bill classifier
sh([
    sys.executable,
    "scripts/25_eval_bill_classifier.py",
    "--checkpoint",
    "outputs/models/bill_classifier_local.pt",
    "--val-dir",
    "data/processed/bill_cutouts/val",
])

## 14) Quantize classifiers (TorchScript INT8)

Creates CPU-friendly TorchScript artifacts using dynamic INT8 quantization on Linear layers.

In [ ]:
# Coin classifier quantized TorchScript
sh([
    sys.executable,
    "scripts/26_quantize_classifier_torchscript.py",
    "--checkpoint",
    "outputs/models/coin_classifier_local.pt",
    "--out",
    "outputs/models/coin_classifier_quantized.ts.pt",
    "--meta-out",
    "outputs/models/coin_classifier_quantized.meta.json",
])

# Bill classifier quantized TorchScript
sh([
    sys.executable,
    "scripts/26_quantize_classifier_torchscript.py",
    "--checkpoint",
    "outputs/models/bill_classifier_local.pt",
    "--out",
    "outputs/models/bill_classifier_quantized.ts.pt",
    "--meta-out",
    "outputs/models/bill_classifier_quantized.meta.json",
])

## 15) Train spoof-guard classifier

Trains the spoof/screen-attack guard that rejects photos-of-screens before they reach the detector.

- Script: `scripts/28_train_spoof_guard.py`
- Output: `outputs/models/spoof_guard_local.pt`

In [ ]:
sh([
    sys.executable,
    "scripts/28_train_spoof_guard.py",
    "--data-dir",
    "data/processed/spoof_guard",
    "--output",
    "outputs/models/spoof_guard_local.pt",
    "--epochs",
    "15",
    "--batch-size",
    "32",
])

## 16) Evaluate spoof-guard classifier

Runs confusion-matrix evaluation on the spoof-guard checkpoint.

In [ ]:
sh([
    sys.executable,
    "scripts/29_eval_spoof_guard.py",
    "--checkpoint",
    "outputs/models/spoof_guard_local.pt",
    "--val-dir",
    "data/processed/spoof_guard/val",
])

## 17) Prepare screen-guard dataset + submit SageMaker training job

### 17a) Build the screen-detection binary dataset from COCO128

Creates a YOLO-format dataset for training the dedicated screen-guard detector on SageMaker.

### 17b) Submit SageMaker training job

Requires AWS credentials. Launches a managed YOLO training job on `ml.g4dn.xlarge`.

In [ ]:
# 17a) Prepare screen-guard dataset
sh([
    sys.executable,
    "scripts/30_prepare_screen_dataset_from_coco128.py",
    "--out",
    "data/processed/screen_guard_dataset",
])

# 17b) Submit SageMaker training job (requires AWS credentials)
# Uncomment to submit:
# sh([
#     sys.executable,
#     "scripts/31_submit_sagemaker_screen_yolo.py",
#     "--data-s3",
#     "s3://your-bucket/screen-guard-data/",
#     "--region",
#     "us-east-1",
# ])

## 18) Download champion models from SageMaker (optional)

If you have AWS credentials configured, download the best-performing model artifacts from completed SageMaker training jobs.

Current champion models in `training/models/`:

| Model | File | Date |
|-------|------|------|
| Bill classifier | `champ-bill-20260223-225355__model.pt` | 2026-02-23 |
| Coin classifier | `champ-coin-20260223-223033__model.pt` | 2026-02-23 |
| Detector | `champ-detector-20260224-185203__model.pt` | 2026-02-24 |
| Screen-guard | `screen-guard-detector-20260316-064336__model.pt` | 2026-03-16 |

In [ ]:
# Download champion models from SageMaker (requires AWS credentials)
# sh([
#     sys.executable,
#     "scripts/40_download_sagemaker_champions.py",
#     "--region",
#     "us-east-1",
#     "--job",
#     "champ-detector-YYYYMMDD-HHMMSS",
# ])

## 19) Docker Compose: Inference API + React Native UI

Start the multi-model inference API with the React Native Expo web frontend.

| URL | Description |
|-----|-------------|
| `http://localhost:8080` | React Native Expo web UI (camera + detection) |
| `http://localhost:8080/rn` | Legacy web dashboard |
| `http://localhost:8080/docs` | Swagger / OpenAPI docs |
| `http://localhost:8080/api/health` | Health check |
| `http://localhost:8080/api/pipeline/infer` | POST inference endpoint |

The inference pipeline runs: **Spoof Guard → Detector → Coin Reader → Bill Reader → Display Names**

In [ ]:
# Start inference API (detached)
sh(["docker", "compose", "up", "-d", "api"])

# Run E2E tests against the API (optional)
# sh(["docker", "compose", "--profile", "test", "up", "--abort-on-container-exit"])

# Tear down
# sh(["docker", "compose", "down"])

## Notes

### Pipeline architecture

The full inference pipeline (served by `training/app/main.py`) is:

1. **Spoof Guard** — Rejects photos-of-screens / photocopied bills before detection.
2. **Detector (YOLO26)** — Localizes banknotes and coins in the image.
3. **Coin Reader** — Classifies cropped coin detections into denominations.
4. **Bill Reader** — Classifies cropped banknote detections into denominations.
5. **Display Names** — Maps internal class IDs to human-readable currency labels (e.g. "CAD $20").

### Current models

Four champion models are deployed in `training/models/`:
- `champ-detector-20260224-185203__model.pt` — YOLO26 banknote + coin detector
- `champ-coin-20260223-223033__model.pt` — ResNet18 coin classifier
- `champ-bill-20260223-225355__model.pt` — ResNet18 bill classifier
- `screen-guard-detector-20260316-064336__model.pt` — YOLO screen-guard detector (SageMaker trained)

### Key directories

| Directory | Contents |
|-----------|----------|
| `scripts/` | Numbered data-pipeline & training scripts (01–40) |
| `configs/` | Label maps, universe links |
| `training/app/` | FastAPI inference API + React Native Expo UI |
| `training/models/` | Champion model weights |
| `data/` | Raw + processed datasets (git-ignored) |
| `outputs/` | Training outputs, reports (git-ignored) |
| `documentation/` | Workflow guides |

### Smoke tests (no datasets required)

The `training/` directory includes `--smoke` flags for quick validation without real data.

In [ ]:
# Quick smoke check (no datasets required)
sh([sys.executable, "training/train_detector.py", "--smoke"])
sh([sys.executable, "training/eval_detector.py", "--smoke"])
sh([sys.executable, "training/coin_classifier/train_coin_classifier.py", "--smoke"])
sh([sys.executable, "training/coin_classifier/eval_coin_classifier.py", "--smoke"])
sh([sys.executable, "training/coin_classifier/infer_coin_classifier.py", "--smoke"])

## (Optional) List outputs

Shows what the pipeline generated locally.

In [ ]:
for path in [Path("data"), Path("outputs")]:
    if not path.exists():
        print(f"missing={path}")
        continue
    print(f"\n{path}/")
    for p in sorted(path.rglob("*")):
        if p.is_file():
            print(p.as_posix())